In [1]:
# Teaching_Video_Generation video eval runner
#
# Target videos:
# - /Users/alyssa/Desktop/education_video/001-remotion-video.mp4
# - /Users/alyssa/Desktop/education_video/002-remotion-video.mp4
# - /Users/alyssa/Desktop/education_video/003-remotion-video.mp4
#
# This notebook supports Colab and local (eduv conda env).

In [2]:
# 1) Environment setup (Colab/local)

import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = False
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    # ffmpeg is required by teachgen.eval.video
    subprocess.check_call(["apt-get", "update", "-qq"])
    subprocess.check_call(["apt-get", "install", "-y", "-qq", "ffmpeg"])

WORK_ROOT = Path("/content/education_video") if IN_COLAB else Path("/Users/alyssa/Desktop/education_video")
WORK_ROOT.mkdir(parents=True, exist_ok=True)

# Python deps
pip_base = [sys.executable, "-m", "pip", "install", "-U"]
if not IN_COLAB:
    pip_base = [sys.executable, "-m", "pip", "install", "--user", "--break-system-packages", "-U"]

subprocess.check_call(pip_base + ["openai", "json-repair", "pandas", "openpyxl"])

# Install project deps
project_dir = WORK_ROOT / "Teaching_Video_Generation"
if project_dir.exists():
    subprocess.check_call(pip_base + ["-r", str(project_dir / "requirements.txt")])

print("IN_COLAB:", IN_COLAB)
print("WORK_ROOT:", WORK_ROOT)
print("PROJECT_DIR exists:", project_dir.exists())

IN_COLAB: True
WORK_ROOT: /content/education_video
PROJECT_DIR exists: False


In [3]:
# 2) Paths + Drive integration + auto-discover final videos

import shutil
from datetime import datetime

DRIVE_INPUT_DIR = None
DRIVE_OUTPUT_ROOT = None

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
    DRIVE_INPUT_DIR = Path("/content/drive/MyDrive/education_video")
    DRIVE_OUTPUT_ROOT = DRIVE_INPUT_DIR / "eval_outputs"
    DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

    # Only use Drive sources in Colab
    if not project_dir.exists():
        drive_project = DRIVE_INPUT_DIR / "Teaching_Video_Generation"
        drive_zip = DRIVE_INPUT_DIR / "Teaching_Video_Generation.zip"

        if drive_project.exists():
            print("Using project folder from Drive:", drive_project)
            shutil.copytree(drive_project, project_dir)
        elif drive_zip.exists():
            print("Using project zip from Drive:", drive_zip)
            shutil.unpack_archive(str(drive_zip), str(WORK_ROOT))
            nested = WORK_ROOT / "Teaching_Video_Generation"
            if not nested.exists():
                raise FileNotFoundError(f"Unzipped project not found at {nested}")
        else:
            raise FileNotFoundError(
                "Missing project in Drive. Put either folder 'Teaching_Video_Generation' "
                "or file 'Teaching_Video_Generation.zip' under MyDrive/education_video."
            )

PROJECT_DIR = project_dir
if not PROJECT_DIR.exists():
    raise FileNotFoundError(f"Project folder not found: {PROJECT_DIR}")

subprocess.check_call(pip_base + ["-r", str(PROJECT_DIR / "requirements.txt")])

# Find 4 folders whose names end with 'outputs'.
SCAN_ROOT = DRIVE_INPUT_DIR if DRIVE_INPUT_DIR is not None else WORK_ROOT
if SCAN_ROOT is None or not SCAN_ROOT.exists():
    raise FileNotFoundError(f"Scan root not found: {SCAN_ROOT}")

candidate_output_dirs = [
    p for p in SCAN_ROOT.rglob("*")
    if p.is_dir()
    and (p.name.lower().endswith("outputs") or p.name.lower().endswith("_output"))
    and p.name.lower() != "eval_outputs"
]

candidate_output_dirs = sorted(candidate_output_dirs, key=lambda p: p.stat().st_mtime, reverse=True)
seen_names = set()
TARGET_OUTPUT_DIRS = []
for p in candidate_output_dirs:
    if p.name in seen_names:
        continue
    seen_names.add(p.name)
    TARGET_OUTPUT_DIRS.append(p)

if not TARGET_OUTPUT_DIRS:
    raise FileNotFoundError(
        f"No '*outputs' folders found under: {SCAN_ROOT}. "
        "Please check Drive path and folder names."
    )

# Keep only likely final videos (exclude intermediate render artifacts).
EXCLUDE_PARTS = {
    "partial_movie_files",
    "chunks",
    "chunk_analyses",
    "media",
    "images",
    "frames",
    "cache",
    "tmp",
}
PREFER_NAME_HINTS = ("final", "remotion-video", "_video", "-video")
MAX_VIDEOS_PER_OUTPUT = 0  # 0 = no limit, evaluate all discovered videos


def _is_intermediate_video(path: Path) -> bool:
    parts = {x.lower() for x in path.parts}
    if parts & EXCLUDE_PARTS:
        return True
    name = path.name.lower()
    # Typical manim/c2v intermediate naming patterns
    if name.startswith("section") and name.endswith(".mp4"):
        return True
    return False


def _video_rank(base_dir: Path, path: Path):
    rel = path.relative_to(base_dir)
    stem = path.stem.lower()
    is_preferred = any(h in stem for h in PREFER_NAME_HINTS)
    return (
        0 if is_preferred else 1,
        len(rel.parts),
        -path.stat().st_mtime,
        path.name.lower(),
    )


EVAL_ITEMS = []
for out_dir in TARGET_OUTPUT_DIRS:
    all_mp4s = [p for p in out_dir.rglob("*.mp4") if p.is_file()]
    filtered_mp4s = [p for p in all_mp4s if not _is_intermediate_video(p)]

    # If filtering is too strict for a folder, fallback to all mp4s.
    pool = filtered_mp4s if filtered_mp4s else all_mp4s
    ranked = sorted(pool, key=lambda p: _video_rank(out_dir, p))
    selected = ranked[:MAX_VIDEOS_PER_OUTPUT] if MAX_VIDEOS_PER_OUTPUT > 0 else ranked

    print(
        f"{out_dir.name}: all={len(all_mp4s)}, filtered={len(filtered_mp4s)}, selected={len(selected)}"
    )
    for mp4 in selected:
        baseline = out_dir.name.replace("_outputs", "").replace("_output", "")
        EVAL_ITEMS.append(
            {
                "source_output_dir": str(out_dir),
                "baseline": baseline,
                "video_path": str(mp4),
                "relative_video_path": str(mp4.relative_to(out_dir)),
            }
        )

if not EVAL_ITEMS:
    raise FileNotFoundError(
        f"No .mp4 files found in selected output folders: {[str(x) for x in TARGET_OUTPUT_DIRS]}"
    )

RUN_ID = datetime.now().strftime("run_%Y%m%d_%H%M%S")
OUTPUT_ROOT = WORK_ROOT / "eval_outputs" / RUN_ID
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("SCAN_ROOT:", SCAN_ROOT)
print("TARGET_OUTPUT_DIRS:")
for d in TARGET_OUTPUT_DIRS:
    print("-", d)
print("Total videos to evaluate:", len(EVAL_ITEMS))
print("OUTPUT_ROOT:", OUTPUT_ROOT)

Mounted at /content/drive
Using project folder from Drive: /content/drive/MyDrive/education_video/Teaching_Video_Generation
c2v_outputs: all=549, filtered=3, selected=3
remotion_outputs: all=3, filtered=3, selected=3
manim_outputs: all=149, filtered=3, selected=3
tm_outputs: all=2, filtered=2, selected=2
PROJECT_DIR: /content/education_video/Teaching_Video_Generation
SCAN_ROOT: /content/drive/MyDrive/education_video
TARGET_OUTPUT_DIRS:
- /content/drive/MyDrive/education_video/c2v_outputs
- /content/drive/MyDrive/education_video/remotion_outputs
- /content/drive/MyDrive/education_video/manim_outputs
- /content/drive/MyDrive/education_video/tm_outputs
Total videos to evaluate: 11
OUTPUT_ROOT: /content/education_video/eval_outputs/run_20260708_091524


In [4]:
# 3) OpenAI official API config

import sys
sys.path.insert(0, "/content/drive/MyDrive/education_video/baseline/runs")
from colab_bootstrap import ensure_openai_key, setup_api_tracking, openai_model, print_api_usage

ensure_openai_key()
setup_api_tracking()
OPENAI_MODEL = openai_model()
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

print("Model:", OPENAI_MODEL)
print("API key set:", bool(OPENAI_API_KEY))
print_api_usage("eval start")

Model: gpt-5
API key set: True


In [7]:
# 4) Patch evaluator LLM backends (single source of truth)

import base64
import json
import subprocess
import tempfile
from pathlib import Path
from typing import TypeVar

from openai import OpenAI
from pydantic import BaseModel
from json_repair import repair_json

OutputModel = TypeVar("OutputModel", bound=BaseModel)


class OfficialTextLLM:
    def __init__(self, model: str = OPENAI_MODEL):
        self.model = model
        self.client = OpenAI(api_key=OPENAI_API_KEY, base_url="https://api.openai.com/v1")

    def analyze(self, prompt: str, output_model: type[OutputModel]) -> OutputModel:
        schema_json = json.dumps(output_model.model_json_schema(), ensure_ascii=False)
        prompt_with_schema = prompt + "\n\nReturn ONLY JSON matching this schema:\n" + schema_json
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt_with_schema}],
        )
        answer = (response.choices[0].message.content or "").strip()
        answer = repair_json(answer)
        return output_model.model_validate_json(answer)


class OfficialVideoLLM:
    def __init__(self, model: str = OPENAI_MODEL):
        self.model = model
        self.client = OpenAI(api_key=OPENAI_API_KEY, base_url="https://api.openai.com/v1")

    def _extract_frames(self, video_path: Path, frame_count: int = 6) -> list[Path]:
        temp_dir = Path(tempfile.mkdtemp(prefix="eval_frames_"))
        pattern = temp_dir / "frame_%03d.jpg"
        cmd = [
            "ffmpeg", "-y", "-i", str(video_path),
            "-vf", "fps=1,scale=960:-1:force_original_aspect_ratio=decrease",
            "-vframes", str(frame_count), str(pattern),
        ]
        subprocess.run(cmd, check=True, capture_output=True)
        return sorted(temp_dir.glob("frame_*.jpg"))

    def analyze(self, video_path: str | Path, prompt: str, output_model: type[OutputModel]) -> OutputModel:
        path = Path(video_path)
        if not path.is_file():
            raise FileNotFoundError(f"Video not found: {path}")

        schema_json = json.dumps(output_model.model_json_schema(), ensure_ascii=False)
        prompt_with_schema = prompt + "\n\nReturn ONLY JSON matching this schema:\n" + schema_json

        # Preferred path: official chat.completions file content input.
        try:
            b64 = base64.b64encode(path.read_bytes()).decode("utf-8")
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[{
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt_with_schema},
                        {"type": "file", "file": {"filename": path.name, "file_data": f"data:video/mp4;base64,{b64}"}},
                    ],
                }],
            )
            answer = (response.choices[0].message.content or "").strip()
            answer = repair_json(answer)
            return output_model.model_validate_json(answer)
        except Exception:
            # Fallback: sample frames and send as image inputs.
            frames = self._extract_frames(path)
            content = [{"type": "text", "text": prompt_with_schema}]
            for frame in frames:
                fb64 = base64.b64encode(frame.read_bytes()).decode("utf-8")
                content.append({"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{fb64}"}})

            response = self.client.chat.completions.create(
                model=self.model,
                messages=[{"role": "user", "content": content}],
            )
            answer = (response.choices[0].message.content or "").strip()
            answer = repair_json(answer)
            return output_model.model_validate_json(answer)


print("LLM backends ready (single patched version).")

LLM backends ready (single patched version).


In [8]:
# 5) Run video evaluation for discovered .mp4 files

import os
import re
import sys
import traceback
from pathlib import Path

if not PROJECT_DIR.exists():
    raise FileNotFoundError(f"Project folder not found: {PROJECT_DIR}")

sys.path.insert(0, str(PROJECT_DIR))

from teachgen.eval.models import EvaluationRequest
import teachgen.eval.run_evaluation as reval

# monkeypatch evaluator backends
reval.TextLLM = OfficialTextLLM
reval.VideoLLM = OfficialVideoLLM


def safe_name(text: str, max_len: int = 96) -> str:
    out = re.sub(r"[^a-zA-Z0-9._-]+", "_", text).strip("_")
    return out[:max_len] if len(out) > max_len else out


results = []

for idx, item in enumerate(EVAL_ITEMS, start=1):
    video_path = Path(item["video_path"])
    source_output_dir = item["source_output_dir"]
    relative_video_path = item["relative_video_path"]

    folder_tag = safe_name(Path(source_output_dir).name)
    file_tag = safe_name(video_path.stem)
    out_dir = OUTPUT_ROOT / f"{idx:03d}_{folder_tag}_{file_tag}"
    out_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 80)
    print(f"[{idx}/{len(EVAL_ITEMS)}] Evaluating: {video_path}")

    if not video_path.exists():
        msg = f"Missing video: {video_path}"
        print("FAILED:", msg)
        results.append(
            {
                "source_output_dir": source_output_dir,
                "baseline": item.get("baseline"),
                "relative_video_path": relative_video_path,
                "video": str(video_path),
                "status": "failed",
                "error": msg,
                "output_dir": str(out_dir),
            }
        )
        continue

    req = EvaluationRequest(video_path=str(video_path))

    try:
        result_path = reval.run_evaluation(
            request=req,
            output_dir=out_dir,
            chunk_seconds=900,
        )
        print("Saved:", result_path)
        results.append(
            {
                "source_output_dir": source_output_dir,
                "baseline": item.get("baseline"),
                "relative_video_path": relative_video_path,
                "video": str(video_path),
                "status": "ok",
                "result": str(result_path),
                "output_dir": str(out_dir),
            }
        )
    except Exception as e:
        tb = traceback.format_exc()
        (out_dir / "error.txt").write_text(tb, encoding="utf-8")
        print("FAILED:", e)
        results.append(
            {
                "source_output_dir": source_output_dir,
                "baseline": item.get("baseline"),
                "relative_video_path": relative_video_path,
                "video": str(video_path),
                "status": "failed",
                "error": f"{type(e).__name__}: {e}",
                "output_dir": str(out_dir),
            }
        )

summary_path = OUTPUT_ROOT / "summary.json"
summary_path.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
print("\nSummary:", summary_path)


[1/11] Evaluating: /content/drive/MyDrive/education_video/c2v_outputs/20260706_150913/CASES/c2v_openrouter_gpt55/1-Fundamentals_of_Personal_Cybersecurity_and_Defense/Fundamentals_of_Personal_Cybersecurity_and_Defense.mp4
[skip] video chunk /content/education_video/eval_outputs/run_20260708_091524/001_c2v_outputs_Fundamentals_of_Personal_Cybersecurity_and_Defense/chunks/chunk_000.mp4
[run] chunk extraction /content/education_video/eval_outputs/run_20260708_091524/001_c2v_outputs_Fundamentals_of_Personal_Cybersecurity_and_Defense/chunk_analyses/chunk_000.json
[run] section inference /content/education_video/eval_outputs/run_20260708_091524/001_c2v_outputs_Fundamentals_of_Personal_Cybersecurity_and_Defense/sections/section_000.json
[run] lecture inference /content/education_video/eval_outputs/run_20260708_091524/001_c2v_outputs_Fundamentals_of_Personal_Cybersecurity_and_Defense/lecture.json
[run] content grader /content/education_video/eval_outputs/run_20260708_091524/001_c2v_outputs_Fun

frame_index:  52%|█████▏    | 348/664 [1:14:29<00:03, 96.19it/s, now=None]

MoviePy - Building video /content/education_video/eval_outputs/run_20260708_091524/010_tm_outputs_tm_cs_saf_002/chunks/chunk_000.mp4.
MoviePy - Writing audio in chunk_000TEMP_MPY_wvf_snd.mp4


frame_index:  52%|█████▏    | 348/664 [1:15:36<00:03, 96.19it/s, now=None]

MoviePy - Done.
MoviePy - Writing video /content/education_video/eval_outputs/run_20260708_091524/010_tm_outputs_tm_cs_saf_002/chunks/chunk_000.mp4



frame_index:  52%|█████▏    | 348/664 [1:20:36<00:03, 96.19it/s, now=None]

... (9 outputs left out)

frame_index:  52%|█████▏    | 348/664 [1:34:39<00:03, 96.19it/s, now=None]

MoviePy - Done.
MoviePy - Writing video /content/education_video/eval_outputs/run_20260708_091524/011_tm_outputs_tm_cs_bin_001/chunks/chunk_000.mp4



frame_index:  52%|█████▏    | 348/664 [1:39:09<00:03, 96.19it/s, now=None]

MoviePy - Done !
MoviePy - video ready /content/education_video/eval_outputs/run_20260708_091524/011_tm_outputs_tm_cs_bin_001/chunks/chunk_000.mp4
[run] chunk extraction /content/education_video/eval_outputs/run_20260708_091524/011_tm_outputs_tm_cs_bin_001/chunk_analyses/chunk_000.json
[run] section inference /content/education_video/eval_outputs/run_20260708_091524/011_tm_outputs_tm_cs_bin_001/sections/section_000.json
[run] lecture inference /content/education_video/eval_outputs/run_20260708_091524/011_tm_outputs_tm_cs_bin_001/lecture.json
[run] content grader /content/education_video/eval_outputs/run_20260708_091524/011_tm_outputs_tm_cs_bin_001/content_grades.json
[run] presentation grader /content/education_video/eval_outputs/run_20260708_091524/011_tm_outputs_tm_cs_bin_001/presentation_grades.json
[run] pedagogy grader /content/education_video/eval_outputs/run_20260708_091524/011_tm_outputs_tm_cs_bin_001/pedagogy_grades.json
[write] evaluation result /content/education_video/eval_

In [7]:
# 5.5) Recover previous run from Drive (no re-evaluation needed)

import json
from pathlib import Path

# Mount Drive in a fresh Colab runtime.
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

RUN_NAME = "run_20260708_091524"  # change if needed

# Search multiple possible roots to avoid hardcoded-path misses.
search_roots = [
    Path('/content/drive/MyDrive/education_video/eval_outputs'),
    Path('/content/drive/My Drive/education_video/eval_outputs'),
]

existing_candidates = [p for p in search_roots if p.exists()]
if not existing_candidates:
    raise FileNotFoundError(
        "Drive eval_outputs root not found. Please run cell #2 first (mount + paths), "
        "or check your Drive account in this runtime."
    )

EXISTING_DRIVE_RUN = None
for root in existing_candidates:
    candidate = root / RUN_NAME
    if candidate.exists():
        EXISTING_DRIVE_RUN = candidate
        break

if EXISTING_DRIVE_RUN is None:
    visible = []
    for root in existing_candidates:
        visible.extend([str(p.name) for p in sorted(root.glob('run_*'))[-8:]])
    raise FileNotFoundError(
        f"Run folder '{RUN_NAME}' not found under {existing_candidates}. "
        f"Visible recent runs: {visible}"
    )

summary_json = EXISTING_DRIVE_RUN / "summary.json"
if not summary_json.exists():
    raise FileNotFoundError(f"summary.json not found: {summary_json}")

loaded_results = json.loads(summary_json.read_text(encoding='utf-8'))

# Repair paths after runtime restart: point rows to files under the Drive run folder.
for item in loaded_results:
    old_out_dir = item.get('output_dir')
    out_name = Path(old_out_dir).name if old_out_dir else None
    if out_name:
        drive_out_dir = EXISTING_DRIVE_RUN / out_name
        item['output_dir'] = str(drive_out_dir)

        if item.get('status') == 'ok':
            current_result = Path(item.get('result', '')) if item.get('result') else None
            if not (current_result and current_result.exists()):
                candidate = drive_out_dir / 'evaluation_result.json'
                if candidate.exists():
                    item['result'] = str(candidate)

results = loaded_results
OUTPUT_ROOT = EXISTING_DRIVE_RUN
DRIVE_OUTPUT_ROOT = EXISTING_DRIVE_RUN.parent

ok_count = sum(1 for r in results if r.get('status') == 'ok')
fail_count = sum(1 for r in results if r.get('status') != 'ok')
print('Recovered from:', summary_json)
print('Recovered results:', len(results), '| ok:', ok_count, '| failed:', fail_count)
print('OUTPUT_ROOT set to:', OUTPUT_ROOT)
print('DRIVE_OUTPUT_ROOT set to:', DRIVE_OUTPUT_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Recovered from: /content/drive/MyDrive/education_video/eval_outputs/run_20260708_091524/summary.json
Recovered results: 11 | ok: 11 | failed: 0
OUTPUT_ROOT set to: /content/drive/MyDrive/education_video/eval_outputs/run_20260708_091524
DRIVE_OUTPUT_ROOT set to: /content/drive/MyDrive/education_video/eval_outputs


In [8]:
# 6) Build summary tables with per-metric subscores

import json
import re
from pathlib import Path
import pandas as pd


def metric_col_name(metric: str) -> str:
    slug = re.sub(r"[^a-z0-9]+", "_", metric.lower()).strip("_")
    return f"metric_{slug}"


def read_eval_result(p: Path):
    if not p.exists():
        return None, {}, []

    data = json.loads(p.read_text(encoding="utf-8"))
    overall = data.get("overall_score")

    metric_scores = {}
    metric_rows = []
    for s in data.get("scores", []):
        metric = s.get("metric")
        score = s.get("score")
        if not metric:
            continue
        metric_scores[metric] = score
        metric_rows.append(
            {
                "metric": metric,
                "score": score,
                "confidence": s.get("confidence"),
                "based_on_inferred_context": s.get("based_on_inferred_context"),
                "rationale": s.get("rationale"),
            }
        )

    return overall, metric_scores, metric_rows


if 'results' not in globals():
    raise RuntimeError("`results` not found. Run cell #5 or #5.5 first.")

rows = []
metric_long_rows = []
all_metrics = set()

for item in results:
    row = {
        "baseline": item.get("baseline"),
        "source_output_dir": item.get("source_output_dir"),
        "relative_video_path": item.get("relative_video_path"),
        "video": item.get("video"),
        "status": item.get("status"),
        "output_dir": item.get("output_dir"),
    }

    if item.get("status") == "ok":
        result_json = Path(item["result"])
        overall, metric_scores, metric_rows = read_eval_result(result_json)
        row["overall_score"] = overall
        row["result_json"] = str(result_json)

        for metric, score in metric_scores.items():
            all_metrics.add(metric)
            row[metric_col_name(metric)] = score

        for mr in metric_rows:
            metric_long_rows.append(
                {
                    "baseline": row.get("baseline"),
                    "source_output_dir": row["source_output_dir"],
                    "relative_video_path": row["relative_video_path"],
                    "video": row["video"],
                    "output_dir": row["output_dir"],
                    "result_json": str(result_json),
                    **mr,
                }
            )
    else:
        row["error"] = item.get("error")

    rows.append(row)

summary_df = pd.DataFrame(rows)

# Ensure stable metric column order
metric_cols = [metric_col_name(m) for m in sorted(all_metrics)]
base_cols = [
    "source_output_dir",
    "relative_video_path",
    "video",
    "status",
    "overall_score",
    *metric_cols,
    "output_dir",
    "result_json",
    "error",
]
ordered_cols = [c for c in base_cols if c in summary_df.columns] + [
    c for c in summary_df.columns if c not in base_cols
]
summary_df = summary_df[ordered_cols]

display(summary_df)

# Long-format metric table: one row per (video, metric)
metric_detail_df = pd.DataFrame(metric_long_rows)
if not metric_detail_df.empty:
    display(metric_detail_df)

# Aggregate: metric averages across successful videos
if not metric_detail_df.empty:
    metric_avg_df = (
        metric_detail_df.groupby("metric", as_index=False)["score"]
        .mean()
        .rename(columns={"score": "avg_score"})
        .sort_values("avg_score", ascending=False)
    )
    display(metric_avg_df)
else:
    metric_avg_df = pd.DataFrame(columns=["metric", "avg_score"])

# Per-baseline metric averages (main summary table)
if not metric_detail_df.empty and "baseline" in metric_detail_df.columns:
    baseline_metric_avg = (
        metric_detail_df.groupby(["baseline", "metric"], as_index=False)["score"]
        .mean()
        .rename(columns={"score": "avg_score"})
    )
    pivot_df = baseline_metric_avg.pivot(index="baseline", columns="metric", values="avg_score")
    pivot_df["overall_avg"] = pivot_df.mean(axis=1, numeric_only=True)
    pivot_df = pivot_df.sort_values("overall_avg", ascending=False)
    display(pivot_df)
else:
    baseline_metric_avg = pd.DataFrame(columns=["baseline", "metric", "avg_score"])
    pivot_df = pd.DataFrame()

csv_path = OUTPUT_ROOT / "summary_table.csv"
xlsx_path = OUTPUT_ROOT / "summary_table.xlsx"
metric_detail_csv = OUTPUT_ROOT / "metric_detail_table.csv"
metric_detail_xlsx = OUTPUT_ROOT / "metric_detail_table.xlsx"
metric_avg_csv = OUTPUT_ROOT / "metric_average_table.csv"
metric_avg_xlsx = OUTPUT_ROOT / "metric_average_table.xlsx"

summary_df.to_csv(csv_path, index=False)
summary_df.to_excel(xlsx_path, index=False)
metric_detail_df.to_csv(metric_detail_csv, index=False)
metric_detail_df.to_excel(metric_detail_xlsx, index=False)
metric_avg_df.to_csv(metric_avg_csv, index=False)
metric_avg_df.to_excel(metric_avg_xlsx, index=False)
baseline_metric_avg.to_csv(OUTPUT_ROOT / "baseline_metric_average_long.csv", index=False)
if not pivot_df.empty:
    pivot_df.to_csv(OUTPUT_ROOT / "baseline_metric_average_pivot.csv")
    pivot_df.to_excel(OUTPUT_ROOT / "baseline_metric_average_pivot.xlsx")

print("Saved table CSV:", csv_path)
print("Saved table XLSX:", xlsx_path)
print("Saved metric detail CSV:", metric_detail_csv)
print("Saved metric detail XLSX:", metric_detail_xlsx)
print("Saved metric average CSV:", metric_avg_csv)
print("Saved metric average XLSX:", metric_avg_xlsx)

,source_output_dir,relative_video_path,video,status,overall_score,metric_bloom_alignment,metric_content_accuracy,metric_icap_alignment,metric_learning_adaptation,metric_learning_objective_coverage,metric_logic,metric_multimedia_learning_design,metric_visual_quality,output_dir,result_json
0,/content/drive/MyDrive/education_video/c2v_out...,20260706_150913/CASES/c2v_openrouter_gpt55/1-F...,/content/drive/MyDrive/education_video/c2v_out...,ok,3.750,4,4,5,4,4,4,3,2,/content/drive/MyDrive/education_video/eval_ou...,/content/drive/MyDrive/education_video/eval_ou...
1,/content/drive/MyDrive/education_video/c2v_out...,20260706_150913/CASES/c2v_openrouter_gpt55/0-B...,/content/drive/MyDrive/education_video/c2v_out...,ok,3.625,4,4,5,3,3,4,3,3,/content/drive/MyDrive/education_video/eval_ou...,/content/drive/MyDrive/education_video/eval_ou...
2,/content/drive/MyDrive/education_video/c2v_out...,20260706_150913/CASES/c2v_openrouter_gpt55/2-V...,/content/drive/MyDrive/education_video/c2v_out...,ok,3.375,3,4,5,3,3,3,3,3,/content/drive/MyDrive/education_video/eval_ou...,/content/drive/MyDrive/education_video/eval_ou...
3,/content/drive/MyDrive/education_video/remotio...,run_20260707_182946/003_tm_cs_col_003_Version_...,/content/drive/MyDrive/education_video/remotio...,ok,4.000,4,5,5,4,3,4,3,4,/content/drive/MyDrive/education_video/eval_ou...,/content/drive/MyDrive/education_video/eval_ou...
4,/content/drive/MyDrive/education_video/remotio...,run_20260707_182946/002_tm_cs_saf_002_Fundamen...,/content/drive/MyDrive/education_video/remotio...,ok,4.125,4,5,5,4,4,4,3,4,/content/drive/MyDrive/education_video/eval_ou...,/content/drive/MyDrive/education_video/eval_ou...
5,/content/drive/MyDrive/education_video/remotio...,run_20260707_182946/001_tm_cs_bin_001_Base-2_N...,/content/drive/MyDrive/education_video/remotio...,ok,3.750,4,4,5,3,3,4,3,4,/content/drive/MyDrive/education_video/eval_ou...,/content/drive/MyDrive/education_video/eval_ou...
6,/content/drive/MyDrive/education_video/manim_o...,run_20260707_085557/03_tm_cs_col_003_Version_C...,/content/drive/MyDrive/education_video/manim_o...,ok,2.625,2,4,1,3,2,4,3,2,/content/drive/MyDrive/education_video/eval_ou...,/content/drive/MyDrive/education_video/eval_ou...
7,/content/drive/MyDrive/education_video/manim_o...,run_20260707_085557/02_tm_cs_saf_002_Fundament...,/content/drive/MyDrive/education_video/manim_o...,ok,3.500,3,4,5,3,3,4,3,3,/content/drive/MyDrive/education_video/eval_ou...,/content/drive/MyDrive/education_video/eval_ou...
8,/content/drive/MyDrive/education_video/manim_o...,run_20260707_085557/01_tm_cs_bin_001_Base-2_Nu...,/content/drive/MyDrive/education_video/manim_o...,ok,3.750,4,4,5,4,3,4,3,3,/content/drive/MyDrive/education_video/eval_ou...,/content/drive/MyDrive/education_video/eval_ou...
9,/content/drive/MyDrive/education_video/tm_outputs,20260706_113322/002_tm_cs_saf_002/tm_cs_saf_00...,/content/drive/MyDrive/education_video/tm_outp...,ok,3.250,3,4,2,3,4,4,3,3,/content/drive/MyDrive/education_video/eval_ou...,/content/drive/MyDrive/education_video/eval_ou...


,source_output_dir,relative_video_path,video,output_dir,result_json,metric,score,confidence,based_on_inferred_context,rationale
0,/content/drive/MyDrive/education_video/c2v_out...,20260706_150913/CASES/c2v_openrouter_gpt55/1-F...,/content/drive/MyDrive/education_video/c2v_out...,/content/drive/MyDrive/education_video/eval_ou...,/content/drive/MyDrive/education_video/eval_ou...,Learning Objective Coverage,4,0.78,True,"Within an inferred intro scope, the lesson str..."
1,/content/drive/MyDrive/education_video/c2v_out...,20260706_150913/CASES/c2v_openrouter_gpt55/1-F...,/content/drive/MyDrive/education_video/c2v_out...,/content/drive/MyDrive/education_video/eval_ou...,/content/drive/MyDrive/education_video/eval_ou...,Content Accuracy,4,0.74,True,Core statements are accurate for an introducto...
2,/content/drive/MyDrive/education_video/c2v_out...,20260706_150913/CASES/c2v_openrouter_gpt55/1-F...,/content/drive/MyDrive/education_video/c2v_out...,/content/drive/MyDrive/education_video/eval_ou...,/content/drive/MyDrive/education_video/eval_ou...,Visual Quality,2,0.82,False,Multiple important visual elements are hard to...
3,/content/drive/MyDrive/education_video/c2v_out...,20260706_150913/CASES/c2v_openrouter_gpt55/1-F...,/content/drive/MyDrive/education_video/c2v_out...,/content/drive/MyDrive/education_video/eval_ou...,/content/drive/MyDrive/education_video/eval_ou...,Multimedia Learning Design,3,0.78,False,Some visuals align well with narration and sup...
4,/content/drive/MyDrive/education_video/c2v_out...,20260706_150913/CASES/c2v_openrouter_gpt55/1-F...,/content/drive/MyDrive/education_video/c2v_out...,/content/drive/MyDrive/education_video/eval_ou...,/content/drive/MyDrive/education_video/eval_ou...,Logic,4,0.74,False,The lesson follows a clear progression from pu...
...,...,...,...,...,...,...,...,...,...,...
83,/content/drive/MyDrive/education_video/tm_outputs,20260706_113322/001_tm_cs_bin_001/tm_cs_bin_00...,/content/drive/MyDrive/education_video/tm_outp...,/content/drive/MyDrive/education_video/eval_ou...,/content/drive/MyDrive/education_video/eval_ou...,Multimedia Learning Design,3,0.68,False,Some visuals align well with the narration (OF...
84,/content/drive/MyDrive/education_video/tm_outputs,20260706_113322/001_tm_cs_bin_001/tm_cs_bin_00...,/content/drive/MyDrive/education_video/tm_outp...,/content/drive/MyDrive/education_video/eval_ou...,/content/drive/MyDrive/education_video/eval_ou...,Logic,3,0.66,False,The sequence is broadly coherent (title → OFF/...
85,/content/drive/MyDrive/education_video/tm_outputs,20260706_113322/001_tm_cs_bin_001/tm_cs_bin_00...,/content/drive/MyDrive/education_video/tm_outp...,/content/drive/MyDrive/education_video/eval_ou...,/content/drive/MyDrive/education_video/eval_ou...,Learning Adaptation,3,0.63,True,Target learner profile: Beginners to binary wi...
86,/content/drive/MyDrive/education_video/tm_outputs,20260706_113322/001_tm_cs_bin_001/tm_cs_bin_00...,/content/drive/MyDrive/education_video/tm_outp...,/content/drive/MyDrive/education_video/eval_ou...,/content/drive/MyDrive/education_video/eval_ou...,Bloom Alignment,4,0.66,True,Intended Bloom level (inferred): Understand. T...


,metric,avg_score
1,Content Accuracy,4.090909
2,ICAP Alignment,4.000000
5,Logic,3.818182
0,Bloom Alignment,3.545455
3,Learning Adaptation,3.363636
4,Learning Objective Coverage,3.181818
7,Visual Quality,3.090909
6,Multimedia Learning Design,3.000000


Saved table CSV: /content/drive/MyDrive/education_video/eval_outputs/run_20260708_091524/summary_table.csv
Saved table XLSX: /content/drive/MyDrive/education_video/eval_outputs/run_20260708_091524/summary_table.xlsx
Saved metric detail CSV: /content/drive/MyDrive/education_video/eval_outputs/run_20260708_091524/metric_detail_table.csv
Saved metric detail XLSX: /content/drive/MyDrive/education_video/eval_outputs/run_20260708_091524/metric_detail_table.xlsx
Saved metric average CSV: /content/drive/MyDrive/education_video/eval_outputs/run_20260708_091524/metric_average_table.csv
Saved metric average XLSX: /content/drive/MyDrive/education_video/eval_outputs/run_20260708_091524/metric_average_table.xlsx


In [ ]:
# 7) Copy outputs and summary table to Google Drive

if DRIVE_OUTPUT_ROOT is None:
    print("Drive output root not configured in local mode.")
else:
    drive_run_dir = DRIVE_OUTPUT_ROOT / OUTPUT_ROOT.name

    # If OUTPUT_ROOT is already a Drive run folder, skip folder copy.
    same_dir = OUTPUT_ROOT.resolve() == drive_run_dir.resolve()
    if same_dir:
        print("OUTPUT_ROOT is already in Drive:", OUTPUT_ROOT)
    else:
        if drive_run_dir.exists():
            shutil.rmtree(drive_run_dir)
        shutil.copytree(OUTPUT_ROOT, drive_run_dir)
        print("Copied folder to:", drive_run_dir)

    zip_path = Path(shutil.make_archive(str(OUTPUT_ROOT), "zip", root_dir=OUTPUT_ROOT))
    zip_dest = DRIVE_OUTPUT_ROOT / zip_path.name
    if zip_path.resolve() != zip_dest.resolve():
        shutil.copy2(zip_path, zip_dest)
    print("Zip at:", zip_dest)

    # Also copy latest tables to stable paths for quick access
    latest_csv = DRIVE_OUTPUT_ROOT / "latest_summary_table.csv"
    latest_xlsx = DRIVE_OUTPUT_ROOT / "latest_summary_table.xlsx"
    latest_metric_detail_csv = DRIVE_OUTPUT_ROOT / "latest_metric_detail_table.csv"
    latest_metric_detail_xlsx = DRIVE_OUTPUT_ROOT / "latest_metric_detail_table.xlsx"
    latest_metric_avg_csv = DRIVE_OUTPUT_ROOT / "latest_metric_average_table.csv"
    latest_metric_avg_xlsx = DRIVE_OUTPUT_ROOT / "latest_metric_average_table.xlsx"

    shutil.copy2(OUTPUT_ROOT / "summary_table.csv", latest_csv)
    shutil.copy2(OUTPUT_ROOT / "summary_table.xlsx", latest_xlsx)
    shutil.copy2(OUTPUT_ROOT / "metric_detail_table.csv", latest_metric_detail_csv)
    shutil.copy2(OUTPUT_ROOT / "metric_detail_table.xlsx", latest_metric_detail_xlsx)
    shutil.copy2(OUTPUT_ROOT / "metric_average_table.csv", latest_metric_avg_csv)
    shutil.copy2(OUTPUT_ROOT / "metric_average_table.xlsx", latest_metric_avg_xlsx)
    if (OUTPUT_ROOT / "baseline_metric_average_pivot.csv").exists():
        shutil.copy2(
            OUTPUT_ROOT / "baseline_metric_average_pivot.csv",
            DRIVE_OUTPUT_ROOT / "latest_baseline_metric_average_pivot.csv",
        )
        shutil.copy2(
            OUTPUT_ROOT / "baseline_metric_average_pivot.xlsx",
            DRIVE_OUTPUT_ROOT / "latest_baseline_metric_average_pivot.xlsx",
        )

    print("Latest summary CSV:", latest_csv)
    print("Latest summary XLSX:", latest_xlsx)
    print("Latest metric detail CSV:", latest_metric_detail_csv)
    print("Latest metric detail XLSX:", latest_metric_detail_xlsx)
    print("Latest metric average CSV:", latest_metric_avg_csv)
    print("Latest metric average XLSX:", latest_metric_avg_xlsx)

OUTPUT_ROOT is already in Drive: /content/drive/MyDrive/education_video/eval_outputs/run_20260708_091524
Zip at: /content/drive/MyDrive/education_video/eval_outputs/run_20260708_091524.zip
Latest summary CSV: /content/drive/MyDrive/education_video/eval_outputs/latest_summary_table.csv
Latest summary XLSX: /content/drive/MyDrive/education_video/eval_outputs/latest_summary_table.xlsx
Latest metric detail CSV: /content/drive/MyDrive/education_video/eval_outputs/latest_metric_detail_table.csv
Latest metric detail XLSX: /content/drive/MyDrive/education_video/eval_outputs/latest_metric_detail_table.xlsx
Latest metric average CSV: /content/drive/MyDrive/education_video/eval_outputs/latest_metric_average_table.csv
Latest metric average XLSX: /content/drive/MyDrive/education_video/eval_outputs/latest_metric_average_table.xlsx
